# The Noise Floor of Agent Evaluation — End-to-End Study Notebook (RP-T02A, v2)

**What this is.** The complete RP-T02A pipeline in one notebook: the determinism ladder (L0/L1/L2/L3), content-addressed record/replay keyed on the **full request body**, the variance-decomposition estimator, oracle validation (Gate G0), the MDE surface and rerun-budget calculator, the **EXP4 ablation battery (AB1–AB11)**, the **regression-detection catalog**, the **empirical winner-flip protocol**, trace divergence (paired), the **confirmatory batch-runner**, and a **final adjudication against the outline's pre-registered rules**.

**v2 changes (from the verification audit).** F1/F2/F11: adjudication now encodes the outline's real decision rules (H1 largest-component, H3 ratio ≥ 2, H6 tied to the surveyed-median placeholder). F3: H5 uses paired RNG streams. F4: Mode B replay keys on the canonical hash of the full request body. F6: harness noise is split into named sub-sources (concurrency, sandbox, tool-ordering) so AB1/AB4/AB5 are real toggles; AB7 fallback-guard demo and AB9 mock failure injection added. F7: the calculator's paired form is explicit; the old dead `between` argument is replaced by an optional treatment-effect-heterogeneity term.

**Two run modes, one engine.** Mode A (`synthetic`, default): no key, no GPU; ground-truth components injected; running it **is** the oracle validation (G0). Mode B (`api`): same engine against any OpenAI-compatible endpoint (hosted key, or local vLLM on a GPU pod); a spend guardrail keeps Run-All safe until you set `api_confirm_spend=True`.

Run top to bottom. The export cell writes every CSV, figure, `prereg.json`, and the final adjudication table.

## 0. Environment and configuration

In [ ]:
import os, sys, json, time, uuid, hashlib, sqlite3, math, random, warnings
from dataclasses import dataclass, field, asdict, replace
from collections import defaultdict, deque
import numpy as np
import pandas as pd
import scipy.stats as st
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore", category=RuntimeWarning)

try:
    from rapidfuzz.distance import Levenshtein as _Lev
    def norm_edit(a, b):
        if not a and not b: return 0.0
        return _Lev.distance(a, b) / max(len(a), len(b))
except Exception:
    def norm_edit(a, b):
        n, m = len(a), len(b)
        if n == 0 and m == 0: return 0.0
        d = list(range(m+1))
        for i in range(1, n+1):
            prev, d[0] = d[0], i
            for j in range(1, m+1):
                cur = d[j]
                d[j] = min(d[j]+1, d[j-1]+1, prev + (a[i-1] != b[j-1]))
                prev = cur
        return d[m] / max(n, m)

RESULTS_DIR = os.environ.get("RPT02A_RESULTS", "rpt02a_results")
os.makedirs(RESULTS_DIR, exist_ok=True)
FIG_DIR = os.path.join(RESULTS_DIR, "figures"); os.makedirs(FIG_DIR, exist_ok=True)
print("results ->", os.path.abspath(RESULTS_DIR))

### Provider notes (Mode B)

Any OpenAI-compatible endpoint works. Two hosted providers are useful because **H4 is a claim about
hosted serving as a class**, so an independent second vendor turns one anecdote into a replication.

| | `api_base_url` | `api_model` example |
|---|---|---|
| OpenAI | `https://api.openai.com/v1` | a dated snapshot, e.g. `gpt-4o-mini-2024-07-18` |
| Anthropic (OpenAI SDK compat) | `https://api.anthropic.com/v1/` | a dated Claude snapshot |
| local vLLM / SGLang | `http://localhost:8000/v1` | the served model id |

Compatibility-layer facts that matter here (from Anthropic's OpenAI SDK compatibility docs):
`temperature` is accepted in [0, 1] and values above 1 are capped (the notebook clamps L0 accordingly);
`seed` is ignored, which is itself consistent with H4's premise that hosted evaluation is not seed-pinnable;
`n` must be 1, which matches the one-call-per-attempt design; prompt caching is not available through the
compatibility layer, so provider variance on that arm is measured without cache deflation, the confound the
outline's threats section flags. Anthropic documents this layer as intended for testing and comparing model
capabilities rather than production, which is exactly this use.

In [ ]:
@dataclass
class Config:
    # ---- run mode ----
    run_mode: str = os.environ.get("RPT02A_RUN_MODE", "synthetic")   # 'synthetic' | 'api'
    # ---- api mode (Mode B) ----
    api_base_url: str = os.environ.get("OPENAI_BASE_URL", "https://api.openai.com/v1")
    api_key: str = os.environ.get("OPENAI_API_KEY", "")
    api_model: str = os.environ.get("RPT02A_MODEL", "gpt-4o-mini")   # pin a DATED snapshot for real runs
    api_benchmark: str = os.environ.get("RPT02A_BENCH", "humaneval_plus")   # 'humaneval_plus' | 'mbpp_plus'
    api_temperature_L0: float = 0.7
    api_max_tokens: int = 1024
    api_confirm_spend: bool = os.environ.get("RPT02A_CONFIRM", "1") == "1"      # full confirmatory matrix; set RPT02A_CONFIRM=0 (or False) for a cheap smoke run
    # ---- study size ----
    n_tasks: int = 200
    n_families: int = 20
    replicates_grid: tuple = (1, 5, 20)
    N_grid: tuple = (50, 200)
    R_decomp: int = 20
    R_reference: int = 400
    R_ablation: int = 60
    bootstrap_iters: int = 400
    # ---- synthetic ground truth (Mode A). Defaults set to the provider-dominant regime the
    # paper's H1/H3 bets describe; Study 2 (robustness) shows recovery holds in other regimes too.
    sigma_between: float = 1.10
    within_family_sd: float = 0.35
    sigma_provider: float = 0.85
    temp0_scale: float = 0.18
    sigma_concurrency: float = 0.13      # harness sub-source (AB1)
    sigma_sandbox: float = 0.11          # harness sub-source (AB4)
    sigma_toolorder: float = 0.09        # harness sub-source (AB5)
    judge_flip_p: float = 0.015
    retry_on_fail: bool = True
    mock_failure_rate: float = 0.0       # AB9 sets this temporarily at L3
    # ---- EXP5 placeholders (LOCKED AT G1 from the survey; placeholders flagged) ----
    surveyed_median_delta: float = 0.02          # PLACEHOLDER until EXP5 completes
    surveyed_delta_quartiles: tuple = (0.01, 0.02, 0.04)  # PLACEHOLDER until EXP5 completes
    # ---- protocol sizes ----
    flip_pairs: int = 10
    detection_magnitudes: tuple = (0.02, 0.05, 0.08, 0.12)
    # ---- statistics ----
    alpha: float = 0.05
    power_target: float = 0.80
    master_seed: int = 20260710
    def harness_sources(self):
        return {"h_conc": self.sigma_concurrency, "h_sand": self.sigma_sandbox, "h_tool": self.sigma_toolorder}

CFG = Config()

# ============================ MODE B: EDIT THESE FOUR LINES ============================
# Leave run_mode as "synthetic" for the free validation-only run (Gate G0 + battery, no API calls).
# Set it to "api" to also measure a real model. Everything else has a working default.
#
# CFG.run_mode     = "api"
# CFG.api_base_url = "https://api.openai.com/v1"     # Anthropic: "https://api.anthropic.com/v1/"
# CFG.api_key      = os.environ.get("OPENAI_API_KEY", "")   # prefer the env var; a literal key
#                                                           # here gets saved into the .ipynb file
# CFG.api_model    = "gpt-4o-mini-2024-07-18"        # pin a DATED snapshot
#
# api_confirm_spend defaults to True, so api mode runs the FULL confirmatory matrix.
# For a cheap plumbing check first, set CFG.api_confirm_spend = False (smoke: 8 tasks, R=2).
# =======================================================================================

assert CFG.run_mode in ("synthetic", "api")
if CFG.run_mode == "api":
    assert CFG.api_key, "run_mode='api' but no api_key. Set OPENAI_API_KEY or CFG.api_key."
    print(f"MODE B: {CFG.api_model} @ {CFG.api_base_url} | bench={CFG.api_benchmark} | "
          f"profile={'FULL confirmatory' if CFG.api_confirm_spend else 'smoke (cheap)'}")
else:
    print("MODE A: synthetic validation only. No API calls, no spend. "
          "Set CFG.run_mode='api' above to measure a real model.")
print(json.dumps({k:v for k,v in asdict(CFG).items() if k!="api_key"}, indent=2, default=str))

## 0b. Pre-registration block (Gate G1 scaffold)

The outline's decision rules, encoded as data. The adjudication cells read **only** from this block, so the paper is graded against the pre-registered rubric and nothing else. Items marked PLACEHOLDER are locked at Gate G1 (pilot calibration + EXP5 survey median) before any confirmatory Mode B run.

In [ ]:
PREREG = {
    "alpha": 0.05,
    "holm_primary_family": ["H1", "H2", "H3", "H6"],
    "H1_rule": "provider component is strictly the largest single within-task component "
               "(paper-level: on at least 2 of 3 benchmarks; per run: on this benchmark)",
    "H2_rule": "harness component bootstrap CI excludes zero (lower bound > 0)",
    "H3_ratio_threshold": 2.0,
    "H3_ratio_prediction": 3.0,
    "H6_flip_threshold": 0.20,
    "H6_falsified_below": 0.05,
    "H6_delta": "CFG.surveyed_median_delta (PLACEHOLDER until EXP5 survey median is locked)",
    "H4_rule": "hosted temperature-zero replicate variance > 0 (Mode B hosted L1 cells only)",
    "H5_rule": "paired trace divergence flags the smallest catalog regression at replicate counts "
               "where the paired outcome test is not significant",
}
PREREG_HASH = hashlib.sha256(json.dumps(PREREG, sort_keys=True).encode()).hexdigest()[:16]
print("PREREG hash:", PREREG_HASH)
for k, v in PREREG.items(): print(f"  {k}: {v}")

## 1. Run ledger and content hashing

Every run is keyed by a content hash of its configuration plus a hashed parity manifest. **v2:** `request_hash` now hashes the canonical form of the *full request body* (model, messages, sampling params), which is the outline's replay-keying spec; the synthetic backend's body is a descriptor that fully determines its request, so the same discipline holds in both modes.

In [ ]:
def canonical_json(obj) -> str:
    return json.dumps(obj, sort_keys=True, separators=(",", ":"), default=str)

def sha(obj) -> str:
    return hashlib.sha256(canonical_json(obj).encode()).hexdigest()[:16]

VOLATILE_FIELDS = {"request_id", "timestamp", "trace_id", "run_id", "wallclock"}

def canonical_request(body: dict) -> dict:
    return {k: v for k, v in body.items() if k not in VOLATILE_FIELDS}

def request_hash(body: dict) -> str:
    '''Canonical hash of the normalized request body (the outline's replay key).'''
    return hashlib.sha256(canonical_json(canonical_request(body)).encode()).hexdigest()

class Ledger:
    def __init__(self, path):
        self.conn = sqlite3.connect(path)
        self.conn.execute('''CREATE TABLE IF NOT EXISTS runs(
            run_id TEXT PRIMARY KEY, rung TEXT, arm TEXT, config_hash TEXT,
            parity_hash TEXT, seed INTEGER, fallbacks INTEGER, valid INTEGER, created REAL)''')
        self.conn.execute('''CREATE TABLE IF NOT EXISTS obs(
            run_id TEXT, task_id TEXT, family_id INTEGER, replicate INTEGER,
            outcome REAL, calls INTEGER, cost REAL, faults INTEGER, trace TEXT)''')
        self.conn.commit()
    def log_run(self, run_id, rung, arm, config_hash, parity_hash, seed, fallbacks):
        valid = 1 if fallbacks == 0 else 0   # AB7: any fallback invalidates the run
        self.conn.execute("INSERT OR REPLACE INTO runs VALUES(?,?,?,?,?,?,?,?,?)",
                          (run_id, rung, arm, config_hash, parity_hash, seed, fallbacks, valid, time.time()))
    def log_obs(self, rows):
        self.conn.executemany("INSERT INTO obs VALUES(?,?,?,?,?,?,?,?,?)", rows)
    def commit(self): self.conn.commit()
    def df(self): return pd.read_sql("SELECT * FROM obs", self.conn)
    def runs_df(self): return pd.read_sql("SELECT * FROM runs", self.conn)

_ledger_path = os.path.join(RESULTS_DIR, "ledger.sqlite")
if os.path.exists(_ledger_path):
    print("WARNING: a ledger already exists in this results dir. Runs will ACCUMULATE in it and\n"
          "         the CSV exports will be OVERWRITTEN by this run. Use a fresh RPT02A_RESULTS\n"
          "         per run (e.g. /workspace/results_openai_humaneval) to keep runs separable.")
LEDGER = Ledger(_ledger_path)
def parity_manifest(rung, arm, cfg):
    return {"benchmark": "synthetic" if cfg.run_mode=="synthetic" else cfg.api_model,
            "model": cfg.api_model if cfg.run_mode=="api" else "sim",
            "n_tasks": cfg.n_tasks, "prompt_policy": "fixed-v1",
            "seed_policy": "master+stream", "arm": arm}
print("ledger ready")

## 2. Content-addressed record/replay and the determinism ladder

Replay keys on the request-body hash, never call order. An arm that skips a call never consumes its recording; an arm that adds a call misses **loudly** and the ledger marks the run invalid (AB7). `key_mode='order'` exists only to demonstrate, in AB6, that naive keying corrupts outcomes when call structure varies.

In [ ]:
class ReplayStore:
    def __init__(self, mode="off", key_mode="content"):
        self.mode = mode              # 'off' | 'record' | 'replay'
        self.key_mode = key_mode      # 'content' | 'order'
        self.store = {}
        self.order_log = []
        self.order_ptr = 0
        self.hits = self.records = self.fallbacks = 0
    def get(self, body, produce):
        if self.mode == "record":
            val = produce()
            if self.key_mode == "content": self.store[request_hash(body)] = val
            else: self.order_log.append(val)
            self.records += 1
            return val, "record"
        if self.mode == "replay":
            if self.key_mode == "content":
                k = request_hash(body)
                if k in self.store:
                    self.hits += 1; return self.store[k], "replay"
                self.fallbacks += 1; return produce(), "fallback"    # LOUD; run becomes invalid
            else:
                if self.order_ptr < len(self.order_log):
                    val = self.order_log[self.order_ptr]; self.order_ptr += 1
                    self.hits += 1; return val, "replay"
                self.fallbacks += 1; return produce(), "fallback"
        return produce(), "live"

@dataclass
class RungSpec:
    name: str
    provider: str        # 'live' | 'temp0' | 'replay' | 'mock'
    harness_noise: bool
    judge: str           # 'llm' | 'rule'
    concurrency: bool

LADDER = {
    "L0": RungSpec("L0", "live",   True,  "llm",  True),
    "L1": RungSpec("L1", "temp0",  True,  "llm",  True),
    "L2": RungSpec("L2", "replay", True,  "llm",  True),
    "L3": RungSpec("L3", "mock",   False, "rule", False),
}
print("ladder:", {k: v.provider for k, v in LADDER.items()})

## 3. Model backends

Both backends expose the same two-step interface: `build_request(task, attempt, rung)` returns the request **body** (what gets hashed for replay), and `generate(body, rng)` executes it. For the replay rung, the body is built exactly as the live rung would build it, so hashes match the golden recordings. The synthetic body is a descriptor that fully determines the simulated request; the API body is the literal chat-completions payload (F4 fix).

In [ ]:
def sigmoid(x): return 1.0 / (1.0 + np.exp(-x))

class SyntheticBackend:
    def __init__(self, cfg): self.cfg = cfg
    def provider_sd(self, provider):
        if provider == "temp0": return self.cfg.sigma_provider * self.cfg.temp0_scale
        if provider == "mock":  return 0.0
        return self.cfg.sigma_provider          # 'live' (and what replay pins)
    def build_request(self, task, attempt, rung):
        eff = "live" if rung.provider == "replay" else rung.provider   # replay consumes live recordings
        return {"kind": "generation", "model": "sim", "task": task["id"], "attempt": attempt,
                "temperature": 0.0 if eff == "temp0" else 0.7}
    def generate(self, body, rung, rng_provider):
        eff = "live" if rung.provider == "replay" else rung.provider
        sd = self.provider_sd(eff)
        eps = float(rng_provider.normal(0.0, sd)) if sd > 0 else 0.0
        return {"eps_p": eps, "text": f"soln(task={body['task']},attempt={body['attempt']})"}

class APIBackend:
    def __init__(self, cfg):
        self.cfg = cfg
        from openai import OpenAI
        # High-replicate studies issue thousands of calls; raise SDK retries and timeout so a
        # transient 429/5xx does not abort a confirmatory cell mid-matrix.
        # max_retries=0: our own loop counts rate-limit events instead of letting the SDK hide
        # them in silent backoff (the failure mode that made a throttled run look merely slow).
        self.client = OpenAI(base_url=cfg.api_base_url, api_key=cfg.api_key or "EMPTY",
                             max_retries=0, timeout=120.0)
        try:
            from openai import RateLimitError, APIConnectionError, APITimeoutError, InternalServerError
            self._RL = RateLimitError
            self._RETRY = (APIConnectionError, APITimeoutError, InternalServerError)
        except Exception:
            self._RL = type("RateLimitError", (Exception,), {}); self._RETRY = (Exception,)
    def build_request(self, task, attempt, rung):
        eff = "live" if rung.provider == "replay" else rung.provider
        temp = 0.0 if eff == "temp0" else min(self.cfg.api_temperature_L0, 1.0)  # Claude compat caps at 1.0
        msgs = [{"role": "system", "content": task.get("system", "You are a coding assistant. Return only code.")},
                {"role": "user", "content": task["prompt"]}]
        return {"kind": "generation", "model": self.cfg.api_model, "messages": msgs,
                "temperature": temp, "max_tokens": self.cfg.api_max_tokens, "attempt": attempt}
    def generate(self, body, rung, rng_provider=None, max_attempts=9):
        delay = 1.0
        for _ in range(max_attempts):
            t_call = time.time()
            try:
                r = self.client.chat.completions.create(
                    model=body["model"], messages=body["messages"],
                    temperature=body["temperature"], max_tokens=body["max_tokens"])
                METER.record(time.time() - t_call)
                return {"text": r.choices[0].message.content or ""}
            except self._RL:
                METER.note_429(delay); time.sleep(delay); delay = min(delay * 2, 60.0)
            except self._RETRY:
                METER.note_error(delay); time.sleep(delay); delay = min(delay * 2, 60.0)
        raise RuntimeError("giving up after repeated API failures")

def make_backend(cfg):
    return SyntheticBackend(cfg) if cfg.run_mode == "synthetic" else APIBackend(cfg)
print("backends ready")

## 4. Synthetic oracle and code benchmarks

Family-clustered tasks with latent difficulty (Mode A) and a code-task loader for Mode B (EvalPlus HumanEval+ if installed, tiny built-in set otherwise).

In [ ]:
def make_oracle_tasks(cfg):
    rng = np.random.default_rng(cfg.master_seed ^ 0xABCDEF)
    fam_means = rng.normal(0.0, cfg.sigma_between, size=cfg.n_families)
    tasks = []
    for i in range(cfg.n_tasks):
        fam = i % cfg.n_families
        q = float(fam_means[fam] + rng.normal(0.0, cfg.within_family_sd))
        tasks.append({"id": i, "family": fam, "q": q})
    return tasks

BUILTIN_CODE_TASKS = [
    {"id": "b0", "family": 0, "prompt": "Write a Python function add(a,b) that returns a+b. Return only the function.",
     "entry": "add", "tests": [((1,2),3), ((5,5),10), ((-1,1),0)]},
    {"id": "b1", "family": 0, "prompt": "Write a Python function is_even(n) returning True iff n is even. Return only the function.",
     "entry": "is_even", "tests": [((4,),True), ((7,),False), ((0,),True)]},
    {"id": "b2", "family": 1, "prompt": "Write a Python function rev(s) that reverses a string. Return only the function.",
     "entry": "rev", "tests": [(("abc",),"cba"), (("",),""), (("xy",),"yx")]},
]

def load_code_tasks(cfg, limit=None):
    try:
        bench = getattr(cfg, "api_benchmark", "humaneval_plus")
        if bench == "mbpp_plus":
            from evalplus.data import get_mbpp_plus as _getter
        else:
            from evalplus.data import get_human_eval_plus as _getter; bench = "humaneval_plus"
        data = _getter()
        tasks = []
        for k, v in list(data.items())[:(limit or 50)]:
            tasks.append({"id": k, "family": 0, "prompt": v["prompt"], "entry": v["entry_point"], "evalplus": v})
        return tasks, bench
    except Exception as e:
        print("EvalPlus unavailable (", type(e).__name__, "); using built-in code tasks.")
        return BUILTIN_CODE_TASKS[:(limit or len(BUILTIN_CODE_TASKS))], "builtin"
print("oracle + code loaders ready")

## 5. The reference harness and the ladder engine (v2)

Harness noise is now a **sum of named sub-sources** (concurrency `h_conc`, sandbox `h_sand`, tool-ordering `h_tool`), each with its own RNG stream, each toggleable via `disabled=` — this is what makes AB1/AB4/AB5 real experiments rather than prose. `rng_arm=` lets two arms share RNG streams for **paired** comparisons (the F3 fix behind H5 and the detection catalog). AB9's mock failure injection lives at L3 via `cfg.mock_failure_rate`.

In [ ]:
def stream_rngs(cfg, rung_name, arm, replicate):
    def mk(source):
        h = int(sha({"seed": cfg.master_seed, "rung": rung_name, "arm": arm,
                     "rep": replicate, "src": source}), 16)
        return np.random.default_rng(h % (2**63))
    return {s: mk(s) for s in ["provider", "judge", "fault", "h_conc", "h_sand", "h_tool"]}

def _extract_code(text):
    """Pull code out of a model response: prefer the longest fenced block, else the whole text."""
    import re
    blocks = re.findall(r"```(?:python|py)?\s*\n(.*?)```", text, re.S)
    return max(blocks, key=len) if blocks else text

import signal as _signal
from contextlib import contextmanager as _ctxmgr

EXEC_TIMEOUT_S = 10.0
GOLD_TIMEOUT_S = 30.0

class _CodeTimeout(BaseException):
    """BaseException on purpose: bare `except Exception` in scoring must not swallow it."""

def _on_alarm(signum, frame):
    raise _CodeTimeout()

@_ctxmgr
def _time_limit(seconds):
    if not hasattr(_signal, "SIGALRM"):
        yield; return
    old = _signal.signal(_signal.SIGALRM, _on_alarm)
    _signal.setitimer(_signal.ITIMER_REAL, seconds)
    try:
        yield
    finally:
        _signal.setitimer(_signal.ITIMER_REAL, 0)
        _signal.signal(_signal.SIGALRM, old)

TIMEOUT_COUNT = {"gold": 0, "candidate": 0}
_GOLD_CACHE = {}   # task_id -> [(args, expected), ...]; computed once, reused across replicates

def _gold_cases(task, max_cases=120):
    """Expected outputs from the canonical solution, cached per task.
    Recomputing these per scoring call makes N*R runs quadratically expensive."""
    tid = str(task["id"])
    if tid in _GOLD_CACHE:
        return _GOLD_CACHE[tid]
    import copy
    ep = task["evalplus"]
    gold_ns = {"__name__": "__gold__"}
    try:
        exec(ep["prompt"] + ep["canonical_solution"], gold_ns)
        gold = gold_ns.get(task["entry"])
    except Exception:
        gold = None
    cases = []
    if gold is not None:
        inputs = list(ep.get("base_input") or [])
        extra = list(ep.get("plus_input") or [])
        inputs += extra[: max(0, max_cases - len(inputs))]
        try:
            with _time_limit(GOLD_TIMEOUT_S):
                for args in inputs:
                    try:
                        cases.append((args, gold(*copy.deepcopy(args))))
                    except Exception:
                        continue
        except _CodeTimeout:
            TIMEOUT_COUNT["gold"] += 1
    _GOLD_CACHE[tid] = cases
    return cases

def _score_code(text, task):
    """Score a model completion. Built-in tasks use their inline tests; EvalPlus tasks are
    executed WITH the prompt preamble (imports) prepended and differentially tested against
    cached canonical outputs. Omitting the preamble makes correct solutions fail on NameError
    and collapses the measured pass rate to an artifact."""
    import math, copy
    code = _extract_code(text)
    ep = task.get("evalplus")

    if ep is None:
        try:
            with _time_limit(EXEC_TIMEOUT_S):
                try:
                    ns = {"__name__": "__candidate__"}; exec(code, ns)
                except _CodeTimeout:
                    raise
                except Exception:
                    return 0
                fn = ns.get(task["entry"])
                if fn is None: return 0
                for args, want in task.get("tests", []):
                    try:
                        if fn(*args) != want: return 0
                    except _CodeTimeout:
                        raise
                    except Exception:
                        return 0
                return 1
        except _CodeTimeout:
            TIMEOUT_COUNT["candidate"] += 1; return 0

    preamble = ep["prompt"].split("def " + task["entry"])[0]
    try:
        with _time_limit(EXEC_TIMEOUT_S):
            try:
                cand_ns = {"__name__": "__candidate__"}; exec(preamble + "\n" + code, cand_ns)
            except _CodeTimeout:
                raise
            except Exception:
                cand_ns = {"__name__": "__candidate__"}; exec(code, cand_ns)
    except _CodeTimeout:
        TIMEOUT_COUNT["candidate"] += 1; return 0
    except Exception:
        return 0
    cand = cand_ns.get(task["entry"])
    if cand is None: return 0

    atol = ep.get("atol") or 0
    def eq(a, b):
        if isinstance(a, float) or isinstance(b, float):
            try: return math.isclose(float(a), float(b), rel_tol=1e-6, abs_tol=max(atol, 1e-6))
            except Exception: return a == b
        if isinstance(a, (list, tuple)) and isinstance(b, (list, tuple)):
            return len(a) == len(b) and all(eq(x, y) for x, y in zip(a, b))
        return a == b

    cases = _gold_cases(task)
    if not cases: return 0
    try:
        with _time_limit(EXEC_TIMEOUT_S):
            for args, want in cases:
                try: got = cand(*copy.deepcopy(args))
                except _CodeTimeout: raise
                except Exception: return 0
                if not eq(got, want): return 0
    except _CodeTimeout:
        TIMEOUT_COUNT["candidate"] += 1; return 0
    return 1

def eval_task(task, rung, backend, store, rngs, cfg, disabled=frozenset()):
    '''One task, one replicate, one rung. Returns outcome, calls, faults, trace.'''
    trace = []; calls = 0; faults = 0
    def do_generate(attempt):
        nonlocal calls, faults
        calls += 1
        body = backend.build_request(task, attempt, rung)
        # AB9: scripted failure injection at the mock rung
        if rung.provider == "mock" and cfg.mock_failure_rate > 0 and rngs["fault"].random() < cfg.mock_failure_rate:
            faults += 1; trace.append(f"gen{attempt}:FAULT")
            return {"eps_p": None, "text": "", "fault": True}
        val, src = store.get(body, lambda: backend.generate(body, rung, rngs["provider"]))
        trace.append(f"gen{attempt}:{src}")
        return val

    if cfg.run_mode == "synthetic":
        eps_h = 0.0
        if rung.harness_noise:
            for s, sd in cfg.harness_sources().items():
                if s not in disabled:
                    eps_h += float(rngs[s].normal(0.0, sd))
        g0 = do_generate(0)
        passed = (g0.get("eps_p") is not None) and ((task["q"] + g0["eps_p"] + eps_h) > 0.0)
        if (not passed) and cfg.retry_on_fail:
            g1 = do_generate(1)
            passed = (g1.get("eps_p") is not None) and ((task["q"] + g1["eps_p"] + eps_h) > 0.0)
        outcome = 1 if passed else 0
        if rung.judge == "llm" and rngs["judge"].random() < cfg.judge_flip_p:
            outcome = 1 - outcome
        trace.append("pass" if outcome else "fail")
        return {"outcome": float(outcome), "calls": calls, "cost": float(calls),
                "faults": faults, "trace": tuple(trace)}
    else:
        g0 = do_generate(0)
        outcome = _score_code(g0.get("text", ""), task)
        if outcome == 0 and cfg.retry_on_fail:
            g1 = do_generate(1)
            outcome = _score_code(g1.get("text", ""), task)
        trace.append("pass" if outcome else "fail")
        return {"outcome": float(outcome), "calls": calls, "cost": float(calls),
                "faults": faults, "trace": tuple(trace)}

def record_corpus(tasks, backend, cfg, arm="base"):
    '''Golden recording pass: record the REACHABLE request set (attempts 0 and 1) so legitimate
    call-structure variation replays exactly; anything outside it misses loudly.'''
    store = ReplayStore(mode="record", key_mode="content")
    rngs = stream_rngs(cfg, "GOLDEN", arm, 0)
    live = RungSpec("GOLDEN", "live", True, "llm", True)
    _live = (cfg.run_mode == "api"); _t0 = time.time(); _k = 0
    if _live: METER.start_arm("golden_recording", 2 * len(tasks))
    for t in tasks:
        for attempt in (0, 1):
            body = backend.build_request(t, attempt, live)
            store.get(body, lambda a=attempt, b=body: backend.generate(b, live, rngs["provider"]))
            _k += 1
            if _live and _k % 25 == 0:
                _heartbeat(rung="record")
    if _live:
        print(f"  golden recording complete: {_k} calls in {(time.time()-_t0)/60:.1f} min")
    rep = ReplayStore(mode="replay", key_mode="content"); rep.store = dict(store.store)
    rep.order_log = list(store.order_log)
    return rep

class CallMeter:
    """Per-call telemetry: latency percentiles, explicit 429/error counts, cumulative vs recent
    throughput. The distinction matters: a throttled run can look healthy on cumulative rate early
    in an arm while the recent rate has collapsed."""
    def __init__(self, window=40):
        self.recent = deque(maxlen=window)     # (t_done, latency)
        self.lat = deque(maxlen=400)
        self.calls = 0; self.rate_limit_hits = 0; self.errors = 0; self.retry_sleep_s = 0.0
        self.arm = None; self.done = 0; self.total = 0; self.t_arm = time.time()
        self.t_proc = time.time(); self.arms_done = 0     # whole-run clock, never reset per arm
    def start_arm(self, arm, total):
        if self.arm is not None: self.arms_done += 1
        self.arm, self.total, self.done, self.t_arm = arm, total, 0, time.time()
        self.recent.clear()
    def record(self, latency):
        self.calls += 1; self.done += 1
        self.lat.append(latency); self.recent.append((time.time(), latency))
    def note_429(self, slept):  self.rate_limit_hits += 1; self.retry_sleep_s += slept
    def note_error(self, slept): self.errors += 1; self.retry_sleep_s += slept
    def _pct(self, p):
        if not self.lat: return None
        v = sorted(self.lat); return round(v[min(int(len(v)*p), len(v)-1)], 2)
    def snapshot(self):
        el = max(time.time() - self.t_arm, 1e-6)
        cum = self.done / el
        rec = None
        if len(self.recent) >= 2:
            span = self.recent[-1][0] - self.recent[0][0]
            rec = (len(self.recent) - 1) / span if span > 0 else None
        eta = ((self.total - self.done) / rec / 60) if (rec and rec > 0) else None
        proc_el = time.time() - self.t_proc
        return {"arm": self.arm, "done": self.done, "total": self.total,
                "elapsed_min": round(el / 60, 2),
                "run_elapsed_min": round(proc_el / 60, 2),
                "arms_done_this_process": self.arms_done,
                "calls_this_process": self.calls,
                "duty_cycle": round(self.calls / max(proc_el, 1e-6), 3),
                "rate_cum_per_s": round(cum, 3),
                "rate_recent_per_s": (round(rec, 3) if rec else None),
                "eta_min": (round(eta, 1) if eta else None),
                "lat_p50": self._pct(0.50), "lat_p95": self._pct(0.95),
                "lat_max": (round(max(self.lat), 2) if self.lat else None),
                "rate_limit_hits": self.rate_limit_hits, "errors": self.errors,
                "retry_sleep_min": round(self.retry_sleep_s / 60, 2),
                "throttled": bool(self.rate_limit_hits > 0 or (rec and cum and rec < 0.4 * cum)),
                "scorer_timeouts": dict(TIMEOUT_COUNT) if "TIMEOUT_COUNT" in globals() else {},
                "updated": time.time()}

METER = CallMeter()

def _heartbeat(**kw):
    """Write current status + append a timestamped row to the time series."""
    try:
        snap = METER.snapshot(); snap.update(kw)
        with open(os.path.join(RESULTS_DIR, "progress.json"), "w") as f:
            json.dump(snap, f)
        with open(os.path.join(RESULTS_DIR, "progress_log.jsonl"), "a") as f:
            f.write(json.dumps(snap) + "\n")
    except Exception:
        pass

def tokenize(s):
    import re
    return re.findall(r"[a-z0-9_]+", s.lower())

class BM25:
    def __init__(self, docs, k1=1.5, b=0.75):
        self.docs = [tokenize(d) for d in docs]; self.k1, self.b = k1, b
        self.N = len(docs); self.avg = sum(map(len, self.docs)) / max(self.N, 1)
        self.df = defaultdict(int)
        for d in self.docs:
            for w in set(d): self.df[w] += 1
    def score(self, q, i):
        d = self.docs[i]; L = len(d); s = 0.0
        from collections import Counter
        tf = Counter(d)
        for w in tokenize(q):
            if w not in tf: continue
            idf = math.log(1 + (self.N - self.df[w] + .5) / (self.df[w] + .5))
            s += idf * tf[w] * (self.k1 + 1) / (tf[w] + self.k1 * (1 - self.b + self.b * L / self.avg))
        return s
    def top(self, q, k):
        return sorted(range(self.N), key=lambda i: -self.score(q, i))[:k]

def build_prompt(task_prompt, exemplars, generic, template):
    parts = []
    if template == "rag":
        parts.append("Use the following similar solved problems as guidance.")
    elif template == "closed":
        parts.append("Solve the problem below directly.")
    else:
        parts.append("Solve the task. Reference material, if any, appears below.")
    for ex in exemplars:
        parts.append("### Similar solved problem\n" + ex)
    for gx in generic:
        parts.append("### Generic example\n" + gx)
    parts.append("### Task\n" + task_prompt + "\nReturn only the completed function in a python code block.")
    return "\n\n".join(parts)
print("Mode B retrieval components ready")


def run_cell(tasks, rung_name, backend, cfg, R, arm="base", golden=None, key_mode="content",
             regression=None, disabled=frozenset(), rng_arm=None, judge_override=None):
    '''Execute R replicates over all tasks at one rung. rng_arm shares RNG streams across arms
    for paired designs. judge_override in {"rule","llm"} toggles the judge without editing the ladder.'''
    rung = LADDER[rung_name]
    if judge_override:
        rung = RungSpec(rung.name, rung.provider, rung.harness_noise, judge_override, rung.concurrency)
    rows = []; total_fallbacks = 0
    run_id = f"{rung_name}:{arm}:{uuid.uuid4().hex[:8]}"
    for r in range(R):
        rngs = stream_rngs(cfg, rung_name, rng_arm or arm, r)
        if rung.provider == "replay":
            store = ReplayStore(mode="replay", key_mode=key_mode)
            if golden is not None:
                store.store = dict(golden.store); store.order_log = list(golden.order_log)
        else:
            store = ReplayStore(mode="off", key_mode=key_mode)
        _live = (cfg.run_mode == "api") and rung.provider in ("live", "temp0")
        _t0 = time.time(); _n = 0; _tot = len(tasks) * R
        if _live and r == 0: METER.start_arm(f"{rung_name}:{arm}", _tot)
        for t in tasks:
            tt = t
            if regression and t["id"] in regression["task_ids"] and cfg.run_mode == "synthetic":
                tt = dict(t); tt["q"] = t["q"] - regression["delta"]
            out = eval_task(tt, rung, backend, store, rngs, cfg, disabled=disabled)
            _n += 1
            if _live and (_n % 25 == 0):
                _heartbeat(rung=rung_name)
            rows.append((run_id, str(t["id"]), t["family"], r, out["outcome"], out["calls"],
                         out["cost"], out["faults"], "|".join(out["trace"])))
        total_fallbacks += store.fallbacks
    df = pd.DataFrame(rows, columns=["run_id","task_id","family_id","replicate",
                                     "outcome","calls","cost","faults","trace"])
    LEDGER.log_run(run_id, rung_name, arm, sha(asdict(cfg)),
                   sha(parity_manifest(rung_name, arm, cfg)), cfg.master_seed, total_fallbacks)
    LEDGER.log_obs(list(df.itertuples(index=False, name=None))); LEDGER.commit()
    return df, total_fallbacks
print("ladder engine v2 ready")

## 6. Variance-decomposition estimator

Identification per the outline: harness = within(L2, rule scorer); judge = within(L2) − within(L2, rule); provider = within(L0) − within(L2); temperature-attributable = within(L0) − within(L1). Between-task variance is estimated at L3, the only rung where within-task variance vanishes, so the estimate carries no within/R inflation (and, for binary outcomes, is explicitly the floor-difficulty split — between-task variance is rung-dependent, per the verification audit F7). Uncertainty via nested cluster-aware bootstrap (families, then tasks).

In [ ]:
def per_task_stats(df):
    g = df.groupby("task_id")["outcome"]
    return pd.DataFrame({"task_mean": g.mean(), "task_var": g.var(ddof=1),
                         "family_id": df.groupby("task_id")["family_id"].first()}).reset_index()

def within_var(df):
    return float(per_task_stats(df)["task_var"].mean())

def between_var(df):
    return float(per_task_stats(df)["task_mean"].var(ddof=1))

def decompose(runs):
    wt = {k: within_var(v) for k, v in runs.items()}
    comp = {}
    comp["between_task"] = between_var(runs["L3"]) if "L3" in runs else np.nan
    if "L2" in runs and "L0" in runs: comp["provider"] = max(wt["L0"] - wt["L2"], 0.0)
    if "L2" in runs: comp["harness_plus_judge"] = wt["L2"]
    if "L2_rule" in runs and "L2" in runs:
        comp["harness"] = wt["L2_rule"]
        comp["judge"] = max(wt["L2"] - wt["L2_rule"], 0.0)
    if "L0" in runs and "L1" in runs: comp["temp_attributable"] = max(wt["L0"] - wt["L1"], 0.0)
    comp["_within_by_rung"] = wt
    return comp

def cluster_bootstrap(runs, cfg, iters=None, naive=False):
    '''Nested cluster-aware bootstrap (families -> tasks). naive=True resamples tasks ignoring
    family structure (AB10 comparison arm).'''
    iters = iters or cfg.bootstrap_iters
    base = runs["L2"] if "L2" in runs else list(runs.values())[0]
    fams = sorted(base["family_id"].unique())
    all_tasks = sorted(base["task_id"].unique())
    keys = ["provider","harness_plus_judge","between_task","temp_attributable","harness","judge"]
    acc = defaultdict(list)
    rng = np.random.default_rng(cfg.master_seed ^ 0x5A5A)
    fam_tasks = {f: sorted(base[base["family_id"]==f]["task_id"].unique()) for f in fams}
    for _ in range(iters):
        if naive:
            sel = set(rng.choice(all_tasks, size=len(all_tasks), replace=True))
        else:
            boot_fams = rng.choice(fams, size=len(fams), replace=True)
            sel_tasks = []
            for f in boot_fams:
                tk = fam_tasks[f]
                sel_tasks.extend(rng.choice(tk, size=len(tk), replace=True))
            sel = set(sel_tasks)
        sub = {k: v[v["task_id"].isin(sel)] for k, v in runs.items()}
        c = decompose(sub)
        for k in keys:
            if k in c and not (isinstance(c[k], float) and np.isnan(c[k])): acc[k].append(c[k])
    return {k: (float(np.percentile(a,2.5)), float(np.percentile(a,97.5))) for k,a in acc.items() if a}
print("estimator ready")

## 7. Oracle validation — Gate G0

High-replication reference vs practical-R estimate, bootstrap CIs, ladder identities, zero-fallback check. This cell is the G0 gate condition.

In [ ]:
def build_runs(tasks, backend, cfg, R, golden, include_L2_rule=True):
    runs = {}
    runs["L0"], fb0 = run_cell(tasks, "L0", backend, cfg, R)
    runs["L1"], _   = run_cell(tasks, "L1", backend, cfg, R)
    runs["L2"], fb2 = run_cell(tasks, "L2", backend, cfg, R, golden=golden)
    runs["L3"], _   = run_cell(tasks, "L3", backend, cfg, R)
    if include_L2_rule:
        runs["L2_rule"], _ = run_cell(tasks, "L2", backend, cfg, R, golden=golden,
                                      arm="rule", judge_override="rule")
    return runs, {"L0_fallbacks": fb0, "L2_fallbacks": fb2}

# The validation track (G0, battery, catalog, flip, H5) ALWAYS runs synthetically, regardless of
# run_mode: it is the oracle-validation gate, costs zero API calls, and must pass before Mode B.
SYNTH = replace(CFG, run_mode="synthetic")
tasks = make_oracle_tasks(SYNTH)
backend = make_backend(SYNTH)
golden = record_corpus(tasks, backend, SYNTH)

runs_hi, fb_hi = build_runs(tasks, backend, SYNTH, SYNTH.R_reference, golden)
ref = decompose(runs_hi)
runs_lo, fb_lo = build_runs(tasks, backend, SYNTH, SYNTH.R_decomp, golden)
est = decompose(runs_lo)
ci  = cluster_bootstrap(runs_lo, SYNTH)

print("L2 replay fallbacks (must be 0):", fb_lo["L2_fallbacks"])
rows = []
for k in ["between_task","provider","harness_plus_judge","harness","judge","temp_attributable"]:
    if k in est:
        lo, hi = ci.get(k, (np.nan, np.nan))
        rows.append({"component": k, "reference(hiR)": round(ref.get(k, np.nan), 4),
                     "estimate": round(est[k], 4), "ci_lo": round(lo, 4), "ci_hi": round(hi, 4),
                     "ref_in_CI": (not np.isnan(lo)) and (lo <= ref.get(k, np.nan) <= hi)})
recovery = pd.DataFrame(rows); print(recovery.to_string(index=False))

In [ ]:
wt = est["_within_by_rung"]
g0_checks = {
    "within decreasing L0>=L1": wt["L0"] + 1e-9 >= wt["L1"],
    "within decreasing L1>=L2 (approx)": wt["L1"] + 0.02 >= wt["L2"],
    "within decreasing L2>=L3": wt["L2"] + 1e-9 >= wt["L3"],
    "L3 within ~ 0": wt["L3"] < 0.01,
    "provider >= 0": est.get("provider", 0) >= 0,
    "L2 replay zero fallbacks": fb_lo["L2_fallbacks"] == 0,
    "reference recovered in CI (majority)": recovery["ref_in_CI"].mean() >= 0.6,
}
G0_PASS = all(g0_checks.values())
for k, v in g0_checks.items(): print(("PASS" if v else "FAIL"), "-", k)
print(); print("GATE G0:", "PASS" if G0_PASS else "FAIL")

## 8. MDE and the rerun-budget calculator (paired form, F7 fix)

Comparisons are **paired on a common eval slice**, so shared task difficulty cancels; the paired
variance is `Var(delta_hat) = het/N + 2*within/(N*R)`, where `het` is treatment-effect heterogeneity
across tasks (Var over tasks of the per-task true effect). `het=0` is the homogeneous-effect default;
the detection catalog estimates it empirically where it matters.

In [ ]:
def z(p): return st.norm.ppf(p)

def mde(within, N, R, alpha=0.05, power=0.80, het=0.0):
    var = het / N + 2.0 * within / (N * R)
    return (z(1 - alpha / 2) + z(power)) * math.sqrt(var)

def detection_power(within, N, R, true_delta, alpha=0.05, het=0.0):
    se = math.sqrt(het / N + 2.0 * within / (N * R))
    crit = z(1 - alpha / 2) * se
    return 1 - st.norm.cdf((crit - true_delta) / se) + st.norm.cdf((-crit - true_delta) / se)

def runs_to_detect(true_delta, within, N, power=0.80, alpha=0.05, het=0.0, Rmax=400):
    for R in range(1, Rmax + 1):
        if detection_power(within, N, R, true_delta, alpha, het) >= power:
            return R
    return None

def min_budget_for(delta, within, alpha=0.05, power=0.80, het=0.0, N_grid=None, R_grid=None):
    N_grid = N_grid or [20, 50, 100, 200, 500]; R_grid = R_grid or [1, 2, 3, 5, 10, 20, 50]
    best = None
    for N in N_grid:
        for R in R_grid:
            if mde(within, N, R, alpha, power, het) <= delta:
                cost = N * R
                if best is None or cost < best[2]: best = (N, R, cost)
    return best

within_L0 = wt["L0"]; within_L2 = wt["L2"]; within_L3 = wt["L3"]
print("MDE @ (N=200,R=1)  L0:", round(mde(within_L0, 200, 1), 4), "| L2:", round(mde(within_L2, 200, 1), 4))
print("MDE @ (N=200,R=20) L0:", round(mde(within_L0, 200, 20), 4), "| L2:", round(mde(within_L2, 200, 20), 4))
for d in (0.05, 0.03):
    print(f"budget for delta={d}: L0 ->", min_budget_for(d, within_L0), "| L2 ->", min_budget_for(d, within_L2))

## 9. Trace divergence, paired (H5, F3 fix)

Base and post runs share RNG streams (`rng_arm`), so at L2 the only trace differences are regression-induced call-structure changes. Divergence is exactly zero at delta = 0 and grows with the regression.

In [ ]:
def trace_divergence(dfa, dfb):
    a = dfa.groupby("task_id")["trace"].agg(lambda s: s.iloc[0])
    b = dfb.groupby("task_id")["trace"].agg(lambda s: s.iloc[0])
    common = a.index.intersection(b.index)
    return float(np.mean([norm_edit(a.loc[i], b.loc[i]) for i in common]))

reg_tasks_half = set(t["id"] for t in tasks[: CFG.n_tasks // 2])
h5_rows = []
if True:  # validation track: always synthetic
    for delta in [0.0, 0.02, 0.05, 0.10]:
        base, _ = run_cell(tasks, "L2", backend, SYNTH, 1, golden=golden, arm=f"h5b{delta}", rng_arm="h5")
        post, _ = run_cell(tasks, "L2", backend, SYNTH, 1, golden=golden, arm=f"h5p{delta}", rng_arm="h5",
                           regression={"task_ids": reg_tasks_half, "delta": delta} if delta > 0 else None)
        bm = base.groupby("task_id")["outcome"].mean(); pm = post.groupby("task_id")["outcome"].mean()
        dd = (bm - pm).dropna()
        pv = st.ttest_rel(bm.loc[dd.index], pm.loc[dd.index]).pvalue if delta > 0 else 1.0
        h5_rows.append({"delta": delta, "paired_trace_divergence": round(trace_divergence(base, post), 4),
                        "paired_outcome_p_at_R1": round(float(pv), 4)})
    h5_df = pd.DataFrame(h5_rows); print(h5_df.to_string(index=False))
else:
    h5_df = pd.DataFrame(); print("H5 paired-trace study runs in synthetic mode; Mode B uses the catalog runs.")

## 10. EXP4 — the ablation battery (AB1–AB11)

Each ablation is executed against its stated prediction from the outline's Table 3. Decomposition
ablations toggle one named harness sub-source at L2 (rule scorer) and check the variance drop against
the injected share. Validity ablations demonstrate the instrument enforces its assumptions. Robustness
ablations check the study's own design choices. Predicted shares use the logit-variance ratios;
the pass tolerance (±0.35 absolute on the share) reflects the logit-to-outcome nonlinearity.

In [ ]:
AB = []
harness_srcs = SYNTH.harness_sources()
tot_h_var = sum(v * v for v in harness_srcs.values())

# --- AB1/AB4/AB5: decomposition toggles (plus AB3 from the G0 runs) ---
base_h, _ = run_cell(tasks, "L2", backend, SYNTH, SYNTH.R_ablation, golden=golden,
                     arm="ab_base", judge_override="rule")
base_hvar = within_var(base_h)
for ab_id, src, label in [("AB1", "h_conc", "concurrency"), ("AB4", "h_sand", "sandbox"),
                          ("AB5", "h_tool", "tool-ordering")]:
    tog, _ = run_cell(tasks, "L2", backend, SYNTH, SYNTH.R_ablation, golden=golden,
                      arm=f"ab_{src}", judge_override="rule", disabled=frozenset({src}))
    drop_share = (base_hvar - within_var(tog)) / base_hvar if base_hvar > 0 else np.nan
    pred_share = (harness_srcs[src] ** 2) / tot_h_var
    AB.append({"ID": ab_id, "ablation": f"disable {label}", "observed": round(drop_share, 3),
               "predicted": round(pred_share, 3),
               "pass": bool(drop_share > 0 and abs(drop_share - pred_share) < 0.35)})

# AB3: judge toggle (from G0 runs)
AB.append({"ID": "AB3", "ablation": "rule scorer vs llm judge at L2",
           "observed": round(est["judge"], 4), "predicted": "judge component > 0",
           "pass": bool(est["judge"] > 0)})

# --- AB2: retry policy off vs on (call-structure change; replay must stay clean) ---
cfg_nr = replace(SYNTH, retry_on_fail=False)
nr, fb_nr = run_cell(tasks, "L2", backend, cfg_nr, 10, golden=golden, arm="ab2_noretry")
wr, fb_wr = run_cell(tasks, "L2", backend, SYNTH, 10, golden=golden, arm="ab2_retry")
AB.append({"ID": "AB2", "ablation": "retry off vs on (call multiset changes)",
           "observed": f"mean calls {nr['calls'].mean():.2f} vs {wr['calls'].mean():.2f}; fallbacks {fb_nr}+{fb_wr}",
           "predicted": "call counts differ; zero fallbacks under content keying",
           "pass": bool(wr["calls"].mean() > nr["calls"].mean() and fb_nr == 0 and fb_wr == 0)})

# --- AB6: content vs call-order keying (Proposition 1 demonstration) ---
cdf, fb_c = run_cell(tasks, "L2", backend, SYNTH, 10, golden=golden, key_mode="content", arm="ab6_content")
odf, fb_o = run_cell(tasks, "L2", backend, SYNTH, 10, golden=golden, key_mode="order", arm="ab6_order")
cm = cdf.groupby("task_id")["outcome"].mean(); om = odf.groupby("task_id")["outcome"].mean()
corrupt = float(np.mean(cm.loc[cm.index.intersection(om.index)] != om.loc[cm.index.intersection(om.index)]))
AB.append({"ID": "AB6", "ablation": "content-addressed vs call-order replay",
           "observed": f"fallbacks {fb_c} vs {fb_o}; outcome disagreement {corrupt:.2f}",
           "predicted": "order keying desyncs and corrupts; content keying exact",
           "pass": bool(fb_c == 0 and fb_o > 0 and corrupt > 0.1)})

# --- AB7: fallback discipline (delete one recording; the guard must fire and invalidate) ---
holey = ReplayStore(mode="replay", key_mode="content"); holey.store = dict(golden.store)
holey.store.pop(next(iter(holey.store)))
hdf, fb_h = run_cell(tasks, "L2", backend, SYNTH, 1, golden=holey, arm="ab7_hole")
run_valid = int(LEDGER.runs_df().sort_values("created").iloc[-1]["valid"])
AB.append({"ID": "AB7", "ablation": "deliberate replay miss",
           "observed": f"fallbacks {fb_h}; ledger valid flag {run_valid}",
           "predicted": "fallback counted; run marked invalid",
           "pass": bool(fb_h > 0 and run_valid == 0)})

# --- AB8: oracle recovery (Gate G0) ---
AB.append({"ID": "AB8", "ablation": "oracle recovers injected components",
           "observed": f"G0_PASS={G0_PASS}", "predicted": "recovery within CIs", "pass": bool(G0_PASS)})

# --- AB9: mock failure injection at L3 ---
cfg_f = replace(SYNTH, mock_failure_rate=0.15)
fdf, _ = run_cell(tasks, "L3", backend, cfg_f, 5, arm="ab9_faults")
clean, _ = run_cell(tasks, "L3", backend, SYNTH, 5, arm="ab9_clean")
fault_rate = fdf["faults"].sum() / fdf["calls"].sum()
AB.append({"ID": "AB9", "ablation": "mock timeouts/malformed output at L3",
           "observed": f"injected-fault rate {fault_rate:.3f}; pass-rate {fdf['outcome'].mean():.3f} vs clean {clean['outcome'].mean():.3f}; crashes 0",
           "predicted": "harness survives; failures degrade outcomes in accounting, no crashes",
           "pass": bool(0.05 < fault_rate < 0.30 and fdf["outcome"].mean() < clean["outcome"].mean())})

# --- AB10: cluster-aware vs naive resampling ---
ci_naive = cluster_bootstrap(runs_lo, SYNTH, iters=200, naive=True)
w_c = ci["harness_plus_judge"][1] - ci["harness_plus_judge"][0]
w_n = ci_naive["harness_plus_judge"][1] - ci_naive["harness_plus_judge"][0]
AB.append({"ID": "AB10", "ablation": "cluster-aware vs naive bootstrap CI width",
           "observed": f"cluster {w_c:.4f} vs naive {w_n:.4f}",
           "predicted": "naive narrower (understates uncertainty) or equal",
           "pass": bool(w_c >= w_n * 0.95)})

# --- AB11: design sensitivity (MDE monotone in N and R; conclusions not budget-artifacts) ---
mono_N = all(mde(within_L2, n2, 5) <= mde(within_L2, n1, 5) for n1, n2 in [(50, 200), (20, 100)])
mono_R = all(mde(within_L2, 100, r2) <= mde(within_L2, 100, r1) for r1, r2 in [(1, 5), (5, 20)])
AB.append({"ID": "AB11", "ablation": "N, R sweeps around pre-registered values",
           "observed": f"MDE monotone in N: {mono_N}; in R: {mono_R}",
           "predicted": "monotone; headline conclusions stable across grid", "pass": bool(mono_N and mono_R)})

ablation_battery = pd.DataFrame(AB)
print(ablation_battery.to_string(index=False))
print(); print("battery pass rate:", f"{ablation_battery['pass'].mean():.0%}")

## 11. Regression-detection catalog (EXP2)

Injected regressions at the outline's four magnitudes across two synthetic layers (code: difficulty
shift on a task subset; infrastructure: judge-flip increase). For each (layer, magnitude, rung, R):
paired base/post runs (shared RNG streams) and a paired test; `detected` means p < alpha. The catalog
converts the decomposition into runs-to-detection evidence for H3, and infrastructure regressions
illustrate the mean-invisible case where H5's trace channel matters.

**Note.** Each catalog cell here is a single paired realization (one seed); the occasional non-monotone entry is sampling noise. Confirmatory runs replicate each cell per the pre-registered budget.

In [ ]:
def catalog_pair(layer, mag, rung, R, tag):
    if layer == "code":
        reg = {"task_ids": reg_tasks_half, "delta": mag}
        cfg_post = SYNTH
    else:  # 'infra': judge flip bump (mean-shift is small; variance-visible)
        reg = None
        cfg_post = replace(SYNTH, judge_flip_p=SYNTH.judge_flip_p + mag / 2)
    base, _ = run_cell(tasks, rung, backend, SYNTH, R, golden=golden, arm=f"cat_b_{tag}", rng_arm=f"cat_{tag}")
    post, _ = run_cell(tasks, rung, backend, cfg_post, R, golden=golden, arm=f"cat_p_{tag}",
                       rng_arm=f"cat_{tag}", regression=reg)
    bm = base.groupby("task_id")["outcome"].mean(); pm = post.groupby("task_id")["outcome"].mean()
    dd = (bm - pm).dropna()
    t = st.ttest_rel(bm.loc[dd.index], pm.loc[dd.index])
    return float(dd.mean()), float(t.pvalue), trace_divergence(base, post)

catalog_rows = []
if True:  # validation track: always synthetic
    for layer in ["code", "infra"]:
        for mag in CFG.detection_magnitudes:
            for rung in ["L0", "L2"]:
                for R in [1, 5, 20]:
                    dmean, p, tdiv = catalog_pair(layer, mag, rung, R, f"{layer}{mag}{rung}{R}")
                    catalog_rows.append({"layer": layer, "magnitude": mag, "rung": rung, "R": R,
                                         "obs_drop": round(dmean, 4), "p": round(p, 4),
                                         "detected": bool(p < CFG.alpha),
                                         "trace_div": round(tdiv, 4)})
    catalog = pd.DataFrame(catalog_rows)
    piv = catalog[catalog.layer == "code"].pivot_table(index=["magnitude"], columns=["rung", "R"],
                                                        values="detected", aggfunc="first")
    print("code-layer detection grid (True = detected):"); print(piv.to_string())
    print(); print("infra layer at R=20 (mean-invisible, trace-visible):")
    print(catalog[(catalog.layer == "infra") & (catalog.R == 20)][
        ["magnitude", "rung", "obs_drop", "p", "detected", "trace_div"]].to_string(index=False))
else:
    catalog = pd.DataFrame(); print("Catalog runs in synthetic mode; Mode B uses the batch-runner cells.")

## 12. Empirical winner-flip protocol (EXP2-flip, H6)

Not simulated from a formula: actual pairs of configurations separated by injected deltas at the
surveyed quartiles (PLACEHOLDER values until EXP5), ten independent rerun pairs each at R = 1,
sign-disagreement measured directly. This is the headline experiment.

**Estimand note.** Injections shift task difficulty in logit space; the outcome-space effect (pass-rate points, the units of EXP5's surveyed deltas) is smaller and is therefore **measured and reported per row** (`realized_outcome_delta`). At Gate G1 the injection magnitudes are calibrated so realized outcome-space deltas hit the surveyed quartiles.

In [ ]:
def empirical_flip(delta, n_pairs, R=1, rung="L0"):
    wrong = 0; disagree = 0; realized = []
    for p in range(n_pairs):
        signs = []
        for rerun in ("a", "b"):
            tag = f"fl_{delta}_{p}_{rerun}"
            base, _ = run_cell(tasks, rung, backend, SYNTH, R, arm=f"{tag}_base", rng_arm=f"{tag}b")
            post, _ = run_cell(tasks, rung, backend, SYNTH, R, arm=f"{tag}_post", rng_arm=f"{tag}p",
                               regression={"task_ids": set(t["id"] for t in tasks), "delta": delta})
            d = base["outcome"].mean() - post["outcome"].mean()
            realized.append(d)
            signs.append(np.sign(d) if d != 0 else 1.0)
        wrong += sum(1 for s in signs if s <= 0)
        disagree += int(signs[0] != signs[1])
    return wrong / (2 * n_pairs), disagree / n_pairs, float(np.mean(realized))

flip_rows = []
if True:  # validation track: always synthetic
    for delta in CFG.surveyed_delta_quartiles:
        ws, dis, realized = empirical_flip(delta, CFG.flip_pairs)
        flip_rows.append({"true_delta": delta, "R": 1,
                          "realized_outcome_delta": round(realized, 4),
                          "empirical_wrong_sign_rate": round(ws, 3),
                          "empirical_pair_disagreement": round(dis, 3)})
    flip_emp = pd.DataFrame(flip_rows); print(flip_emp.to_string(index=False))
else:
    flip_emp = pd.DataFrame(); print("Flip protocol runs in the batch-runner for Mode B.")

def winner_flip_analytic(delta, within, N, R, trials=4000, seed=0):
    rng = np.random.default_rng(seed)
    se = math.sqrt(2.0 * within / (N * R))
    a = rng.normal(delta, se, size=trials); b = rng.normal(delta, se, size=trials)
    return float(np.mean(np.sign(a) != np.sign(b)))

## 13. Final adjudication against the pre-registered rules

Reads only from the PREREG block (F1/F2/F11 fix). Mode-aware: Mode A verdicts are demonstrations on
injected ground truth that exercise the exact rules; Mode B verdicts on real models are the paper's
results. H4 requires hosted Mode B cells and is marked accordingly.

In [ ]:
adj = []
def rule(hyp, measured, verdict, scope):
    adj.append({"hypothesis": hyp, "measured": measured, "verdict": verdict, "scope": scope})

scope = "DEMONSTRATION (synthetic ground truth)"  # validation track; Mode B empirical rows are appended by the batch cell

# H1: provider strictly the largest single within-task component
h1 = est["provider"] > max(est["harness"], est["judge"])
rule("H1 provider largest single component",
     f"provider={est['provider']:.4f} vs harness={est['harness']:.4f}, judge={est['judge']:.4f}",
     "PASS" if h1 else "FAIL", scope + "; paper rule aggregates over >=2 of 3 benchmarks")

# H2: harness CI excludes zero
h2_lo = ci.get("harness", (np.nan, np.nan))[0]
rule("H2 harness CI excludes zero", f"CI lower bound = {h2_lo:.4f}",
     "PASS" if h2_lo > 0 else "FAIL", scope)

# H3: runs-to-detect ratio >= 2.0 at the smallest catalog magnitude
d_small = CFG.detection_magnitudes[0]
r0 = runs_to_detect(d_small, within_L0, CFG.n_tasks); r2 = runs_to_detect(d_small, within_L2, CFG.n_tasks)
ratio = (r0 / r2) if (r0 and r2) else np.nan
h3 = bool(ratio >= PREREG["H3_ratio_threshold"]) if not np.isnan(ratio) else False
rule("H3 detection ratio >= 2.0 (pred 3.0)",
     f"runs-to-detect L0={r0}, L2={r2}, ratio={ratio:.2f}",
     "PASS" if h3 else "FAIL", scope)

# H6: empirical flip rate at surveyed-median delta (PLACEHOLDER until EXP5)
if len(flip_emp):
    row = flip_emp[flip_emp.true_delta == CFG.surveyed_median_delta]
    fr = float(row["empirical_wrong_sign_rate"].iloc[0]) if len(row) else np.nan
else:
    fr = np.nan
h6 = bool(fr >= PREREG["H6_flip_threshold"]) if not np.isnan(fr) else False
realized = float(flip_emp[flip_emp.true_delta == CFG.surveyed_median_delta]["realized_outcome_delta"].iloc[0]) if len(flip_emp) else np.nan
rule("H6 flip rate >= 0.20 at surveyed-median delta",
     f"empirical wrong-sign rate @ target={CFG.surveyed_median_delta} (PLACEHOLDER; realized outcome delta={realized}) = {fr}",
     "PASS" if h6 else ("N/A" if np.isnan(fr) else "FAIL"),
     scope + "; delta locked from EXP5 at G1")

# H4: adjudicated empirically by the Mode B batch cell (hosted/api L1 replicates)
rule("H4 hosted temp-0 not deterministic", "adjudicated in the Mode B section from batch L1 cells", "N/A", "Mode B only")

# H5: trace flags smallest catalog regression where outcome test does not (paired)
if len(catalog):
    small = catalog[(catalog.layer == "code") & (catalog.magnitude == d_small) & (catalog.rung == "L2") & (catalog.R == 1)]
    h5 = bool(len(small) and small["trace_div"].iloc[0] > 0 and not small["detected"].iloc[0])
    rule("H5 trace flags before outcome (smallest regression, R=1)",
         f"trace_div={float(small['trace_div'].iloc[0]) if len(small) else np.nan}, outcome detected={bool(small['detected'].iloc[0]) if len(small) else None}",
         "PASS" if h5 else "FAIL", scope)
else:
    rule("H5 trace precedes outcome", "requires catalog runs", "N/A", "catalog")

adjudication_final = pd.DataFrame(adj)
print(adjudication_final.to_string(index=False))
print(); print("NOTE: validation-track verdicts exercise the pre-registered rules on injected ground truth;"
      " the paper's empirical verdicts are the Mode B rows appended below when run_mode='api'.")

## 14. Confirmatory batch-runner

Sweeps the outline's Table 1 confirmatory core (rungs L0/L2/L3; N in {50, 200}; R in {1, 5, 20};
plus hosted L1 in api mode) and writes the publication CSVs. In Mode A this runs in full as the
demonstration. In Mode B it estimates the call count first and refuses to run the full matrix unless
`CFG.api_confirm_spend = True` (the smoke profile runs regardless).

In [ ]:
MODEB = {}

def run_confirmatory_matrix(profile=None):
    global MODEB
    if profile is None:
        profile = "full" if CFG.run_mode == "synthetic" or CFG.api_confirm_spend else "smoke"
    api = CFG.run_mode == "api"
    if api and profile == "smoke":
        print("NOTE: api_confirm_spend is False -> SMOKE profile (8 tasks, R=2, no L1/H4 cells,\n"
              "      no Mode B decomposition, no flip protocol). Set CFG.api_confirm_spend=True\n"
              "      (or env RPT02A_CONFIRM=1) for the confirmatory matrix that the paper needs.")
    bk = make_backend(CFG) if api else backend      # api backend only here; validation track stayed synthetic
    # L3 (scripted mock provider) is a synthetic-track and repo-scale construct: in api mode a
    # "mock rung" would just re-hit the live endpoint, so api batches run L0/L1/L2 only.
    if profile == "smoke":
        Ns, Rs = [8], [2]
        rungs = ["L0", "L2"] if api else ["L0", "L2", "L3"]
    else:
        Ns, Rs = list(CFG.N_grid), list(CFG.replicates_grid)
        rungs = (["L0", "L1", "L2"] if api else ["L0", "L2", "L3"])
    if api:
        bench_tasks, bench_name = load_code_tasks(CFG, limit=max(Ns))
        NT = len(bench_tasks)
        live_rungs = sum(1 for r in rungs if r in ("L0", "L1"))   # L2 replays; L3 is mock
        retry = 1.2 if CFG.retry_on_fail else 1.0
        phases = {
            "golden recording": 2 * NT,
            "batch matrix": sum(min(N, NT) * R for N in Ns for R in Rs) * live_rungs,
        }
        if profile == "full":
            phases["Mode B decomposition"] = NT * max(Rs) * 2          # mb_L0 + mb_L1 (L2/L2_rule replay)
            phases["flip protocol"] = 2 * min(CFG.flip_pairs, 6) * 2 * (min(50, NT) * 2)
        est_calls = int(sum(phases.values()) * retry)
        print(f"benchmark: {bench_name} ({NT} tasks)")
        for k, v in phases.items():
            print(f"    {k:<22} ~{int(v*retry):>7,} calls")
        print(f"    {'TOTAL':<22} ~{est_calls:>7,} live calls")
        print(f"  Rough cost at $0.15/$0.60 per 1M tokens (gpt-4o-mini scale, ~130 in / ~60 out per call):"
              f" ${est_calls*(130*0.15 + 60*0.60)/1e6:.2f}")
        print(f"  WALL CLOCK: calls are sequential. At ~0.6-1.0 s each this is roughly"
              f" {est_calls*0.6/3600:.1f}-{est_calls*1.0/3600:.1f} hours, plus ~15 min of scoring.")
        print(f"  To shrink: lower CFG.N_grid / CFG.replicates_grid, or set"
              f" CFG.api_confirm_spend=False for a smoke check first.")
        # ---- resumable api runs: persist the golden store; checkpoint completed cells ----
        CKPT_DIR = os.path.join(RESULTS_DIR, "cells"); os.makedirs(CKPT_DIR, exist_ok=True)
        CKPT_SIG = sha({"model": CFG.api_model, "bench": bench_name, "NT": len(bench_tasks),
                        "Ns": [int(min(N, len(bench_tasks))) for N in Ns], "Rs": list(Rs),
                        "rungs": list(rungs)})
        GOLD_PATH = os.path.join(RESULTS_DIR, "golden_store.json")
        gold = None
        if os.path.exists(GOLD_PATH):
            try:
                gd = json.load(open(GOLD_PATH))
                if gd.get("sig") == CKPT_SIG:
                    gold = ReplayStore(mode="replay", key_mode="content")
                    gold.store = dict(gd["store"]); gold.order_log = list(gd.get("order_log", []))
                    print(f"RESUMING: golden store loaded ({len(gold.store)} recordings); "
                          f"recording pass skipped", flush=True)
            except Exception:
                gold = None
        if gold is None:
            gold = record_corpus(bench_tasks, bk, CFG)
            try:
                with open(GOLD_PATH, "w") as f:
                    json.dump({"sig": CKPT_SIG, "store": gold.store, "order_log": gold.order_log}, f)
            except Exception as e:
                print(f"  (golden store not persisted: {type(e).__name__})", flush=True)
        _dc = [f[:-5] for f in os.listdir(CKPT_DIR)]
        if _dc:
            print(f"RESUMING: {len(_dc)} matrix cell(s) checkpointed and will be skipped: {sorted(_dc)}",
                  flush=True)

        def _cell_ckpt(name):
            p = os.path.join(CKPT_DIR, f"{name}.json")
            if not os.path.exists(p): return None
            try:
                d = json.load(open(p))
                if d.get("sig") != CKPT_SIG: return None
                import io as _io
                return {r: pd.read_json(_io.StringIO(dj), orient="split") for r, dj in d["runs"].items()}
            except Exception:
                return None

        def _save_cell(name, cellruns):
            try:
                with open(os.path.join(CKPT_DIR, f"{name}.json"), "w") as f:
                    json.dump({"sig": CKPT_SIG,
                               "runs": {r: df.to_json(orient="split") for r, df in cellruns.items()}}, f)
            except Exception as e:
                print(f"  (cell checkpoint failed for {name}: {type(e).__name__})", flush=True)
    else:
        bench_tasks = tasks; gold = golden
    rows = []; last_cellruns = {}
    for N in Ns:
        sub = bench_tasks[:min(N, len(bench_tasks))]
        for R in Rs:
            cellruns = _cell_ckpt(f"cm_{N}_{R}") if api else None
            if cellruns is not None:
                print(f"  cell N={N} R={R}: RESUMED from checkpoint", flush=True)
            else:
                cellruns = {}
                for rung in rungs:
                    _c0 = time.time()
                    cellruns[rung], fb = run_cell(sub, rung, bk, CFG if api else SYNTH, R,
                                                  arm=f"cm_{N}_{R}_{rung}", golden=gold)
                    if api:
                        _to = TIMEOUT_COUNT.get("candidate", 0)
                        print(f"  cell N={len(sub)} R={R} {rung}: {(time.time()-_c0)/60:.1f} min "
                              f"(scorer timeouts so far: {_to})", flush=True)
                if api:
                    _save_cell(f"cm_{N}_{R}", cellruns)
            last_cellruns = cellruns
            for rung in rungs:
                w = within_var(cellruns[rung]) if R >= 2 else np.nan
                rows.append({"N": len(sub), "R": R, "rung": rung,
                             "within": (round(w, 5) if R >= 2 else np.nan),
                             "mean": round(cellruns[rung]["outcome"].mean(), 4),
                             "mde_at_cell": (round(mde(w, len(sub), R), 4) if R >= 2 else np.nan)})
    out = pd.DataFrame(rows)
    out.to_csv(f"{RESULTS_DIR}/confirmatory_decomposition.csv", index=False)
    print(f"profile={profile}; wrote confirmatory_decomposition.csv ({len(out)} rows)")

    if api and profile == "full":
        # Full Mode B decomposition (with L2_rule) at the largest cell, plus natural-pair flip.
        Nmax, Rmax = max(Ns), max(Rs)
        sub = bench_tasks[:min(Nmax, len(bench_tasks))]
        runs_b = {}
        runs_b["L0"], _ = run_cell(sub, "L0", bk, CFG, Rmax, arm="mb_L0", golden=gold)
        runs_b["L1"], _ = run_cell(sub, "L1", bk, CFG, Rmax, arm="mb_L1", golden=gold)
        runs_b["L2"], fb_b = run_cell(sub, "L2", bk, CFG, Rmax, arm="mb_L2", golden=gold)
        runs_b["L2_rule"], _ = run_cell(sub, "L2", bk, CFG, Rmax, arm="mb_L2r", golden=gold, judge_override="rule")
        runs_b["L3"] = runs_b["L2_rule"]   # api floor: replay + rule scorer (deterministic); between-task read here
        MODEB["est"] = decompose(runs_b)
        MODEB["ci"] = cluster_bootstrap(runs_b, CFG, iters=200)
        MODEB["fallbacks"] = fb_b
        # natural comparison pairs for the flip protocol (real configs; deltas are measured, not injected)
        def pair_flip(make_armB, label, n_pairs, N, R=1):
            wrong = disagree = 0; realized = []
            subN = bench_tasks[:min(N, len(bench_tasks))]
            for p in range(n_pairs):
                signs = []
                for rerun in ("a", "b"):
                    tag = f"mbfl_{label}_{p}_{rerun}"
                    a, _ = run_cell(subN, "L0", bk, CFG, R, arm=f"{tag}_A", rng_arm=f"{tag}A", golden=gold)
                    b, _ = make_armB(subN, tag, gold)
                    d = a["outcome"].mean() - b["outcome"].mean()
                    realized.append(d); signs.append(np.sign(d) if d != 0 else 1.0)
                disagree += int(signs[0] != signs[1])
            return {"pair": label, "n_pairs": n_pairs,
                    "realized_delta_mean": round(float(np.mean(realized)), 4),
                    "pair_disagreement": round(disagree / n_pairs, 3)}
        def armB_retryoff(subN, tag, gold):
            cfg_nr = replace(CFG, retry_on_fail=False)
            return run_cell(subN, "L0", bk, cfg_nr, 1, arm=f"{tag}_B", rng_arm=f"{tag}B", golden=gold)
        def armB_temp0(subN, tag, gold):
            return run_cell(subN, "L1", bk, CFG, 1, arm=f"{tag}_B", rng_arm=f"{tag}B", golden=gold)
        n_pairs = min(CFG.flip_pairs, 6)
        MODEB["flip"] = pd.DataFrame([
            pair_flip(armB_retryoff, "retry_on_vs_off", n_pairs, N=50),
            pair_flip(armB_temp0,   "prodtemp_vs_temp0", n_pairs, N=50)])
        MODEB["flip"].to_csv(f"{RESULTS_DIR}/modeb_flip.csv", index=False)
        print("Mode B decomposition + natural-pair flip complete.")
    return out

confirmatory = run_confirmatory_matrix()
confirmatory.head(12)

## 14b. Mode B empirical adjudication

Appended only when `run_mode='api'` with the full profile: the paper's empirical verdicts on this
(benchmark, model) cell, graded against the same PREREG rules. H1/H2 from the real decomposition;
H3 as the measured-within detection-budget ratio at the surveyed-median delta; H4 from hosted/api
L1 replicates; H6 from the natural-pair flip protocol (realized deltas measured, not injected).
Paper-level aggregation (2 of 3 benchmarks, multiple scaffolds) happens across notebook runs.

In [ ]:
if CFG.run_mode == "api" and MODEB.get("est"):
    eb, cb = MODEB["est"], MODEB["ci"]
    wb = eb["_within_by_rung"]
    emp = []
    h1 = eb.get("provider", 0) > max(eb.get("harness", 0), eb.get("judge", 0))
    emp.append({"hypothesis": "H1 provider largest single component",
                "measured": f"provider={eb.get('provider',0):.4f} vs harness={eb.get('harness',0):.4f}, judge={eb.get('judge',0):.4f}",
                "verdict": "PASS" if h1 else "FAIL", "scope": "EMPIRICAL (this benchmark x model)"})
    h2lo = cb.get("harness", (np.nan, np.nan))[0]
    emp.append({"hypothesis": "H2 harness CI excludes zero", "measured": f"CI lower bound = {h2lo:.4f}",
                "verdict": "PASS" if h2lo > 0 else "FAIL", "scope": "EMPIRICAL"})
    r0 = runs_to_detect(CFG.surveyed_median_delta, wb["L0"], CFG.N_grid[-1])
    r2 = runs_to_detect(CFG.surveyed_median_delta, wb["L2"], CFG.N_grid[-1])
    ratio = (r0 / r2) if (r0 and r2) else np.nan
    emp.append({"hypothesis": "H3 detection ratio >= 2.0",
                "measured": f"runs-to-detect L0={r0}, L2={r2}, ratio={ratio if np.isnan(ratio) else round(ratio,2)}",
                "verdict": ("PASS" if (not np.isnan(ratio) and ratio >= PREREG["H3_ratio_threshold"]) else "FAIL"),
                "scope": "EMPIRICAL (measured withins; catalog injection on real scaffolds is repo-scale EXP2)"})
    w_l1 = wb.get("L1", np.nan)
    emp.append({"hypothesis": "H4 hosted temp-0 residual variance > 0",
                "measured": f"within(L1)={w_l1 if np.isnan(w_l1) else round(w_l1,5)}",
                "verdict": ("PASS" if (not np.isnan(w_l1) and w_l1 > 1e-6) else ("FAIL" if not np.isnan(w_l1) else "N/A")),
                "scope": "EMPIRICAL (hosted/api endpoint)"})
    if isinstance(MODEB.get("flip"), pd.DataFrame) and len(MODEB["flip"]):
        for _, r in MODEB["flip"].iterrows():
            emp.append({"hypothesis": "H6 natural-pair winner instability",
                        "measured": f"{r['pair']}: realized delta {r['realized_delta_mean']}, pair disagreement {r['pair_disagreement']}",
                        "verdict": "REPORTED (adjudicated against surveyed bins at G2)",
                        "scope": "EMPIRICAL (natural configuration pairs)"})
    modeb_adj = pd.DataFrame(emp)
    adjudication_all = pd.concat([adjudication_final, modeb_adj], ignore_index=True)
    adjudication_all.to_csv(f"{RESULTS_DIR}/adjudication_final.csv", index=False)
    print(modeb_adj.to_string(index=False))
else:
    print("Synthetic run: Mode B empirical adjudication appears here when run_mode='api' with the full profile.")

## 15. Figures

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
order = [k for k in ["between_task","provider","harness_plus_judge","harness","judge","temp_attributable"] if k in est]
xs = np.arange(len(order))
ref_v = [ref.get(k, np.nan) for k in order]; est_v = [est[k] for k in order]
lo = [max(est[k] - ci.get(k, (est[k], est[k]))[0], 0) for k in order]
hi = [max(ci.get(k, (est[k], est[k]))[1] - est[k], 0) for k in order]
ax.bar(xs - 0.18, ref_v, width=0.36, label="reference (high R)", color="#888")
ax.bar(xs + 0.18, est_v, width=0.36, yerr=[lo, hi], capsize=3, label="estimate (practical R)", color="#3b6")
ax.set_xticks(xs); ax.set_xticklabels(order, rotation=25, ha="right"); ax.set_ylabel("variance of outcome")
ax.set_title("Variance decomposition: estimator recovers ground truth"); ax.legend()
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/fig1_decomposition.png", dpi=140); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
rungs4 = ["L0", "L1", "L2", "L3"]
ax.plot(rungs4, [wt[r] for r in rungs4], "o-", color="#c33")
for r in rungs4: ax.annotate(f"{wt[r]:.3f}", (r, wt[r]), textcoords="offset points", xytext=(0, 8))
ax.set_ylabel("within-task run-to-run variance")
ax.set_title("Determinism ladder: variance falls as sources are removed")
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/fig2_ladder.png", dpi=140); plt.show()

In [ ]:
Rs = list(range(1, 31))
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(Rs, [mde(within_L0, 200, R) for R in Rs], label="L0 (live)", color="#c33")
ax.plot(Rs, [mde(within_L2, 200, R) for R in Rs], label="L2 (replay)", color="#3b6")
ax.set_xlabel("replicates R"); ax.set_ylabel("minimum detectable effect (paired)")
ax.set_title("MDE vs replicate budget (N=200)"); ax.legend()
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/fig3_mde.png", dpi=140); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ds = np.linspace(0.002, 0.08, 20)
for R, col in [(1, "#c33"), (10, "#36b")]:
    ax.plot(ds, [winner_flip_analytic(d, within_L0, 200, R, seed=CFG.master_seed) for d in ds],
            label=f"analytic, R={R}", color=col)
if CFG.run_mode == "synthetic" and len(flip_emp):
    ax.scatter(flip_emp.get("realized_outcome_delta", flip_emp["true_delta"]), flip_emp["empirical_pair_disagreement"],
               color="k", zorder=5, label="empirical protocol (R=1), at realized outcome delta")
ax.axhline(0.05, ls="--", color="#999")
ax.set_xlabel("true effect size"); ax.set_ylabel("pairwise winner-disagreement rate")
ax.set_title("Winners flip at small effect sizes"); ax.legend()
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/fig4_flip.png", dpi=140); plt.show()

## 16. Mode B notes and smoke test

Point the config at any OpenAI-compatible endpoint. Replay now keys on the full request body (F4),
so record/replay is faithful for real requests. Before confirmatory Mode B runs: pin a **dated model
snapshot**, set `api_confirm_spend=True`, and lock the PREREG placeholders at G1 (pilot thresholds,
EXP5 survey median). Code scoring executes generated code in-process: fine on a throwaway pod, not a
hardened sandbox.

In [ ]:
if CFG.run_mode == "api":
    print("Mode B active: the batch-runner above already executed"
          f" (profile={'full' if CFG.api_confirm_spend else 'smoke'}).")
    print("For the full confirmatory matrix set CFG.api_confirm_spend=True and re-run the batch cell.")
else:
    print("Mode A active. For Mode B: set CFG.run_mode='api', endpoint, key, model; optionally"
          " api_confirm_spend=True; then Run All.")

## 17. Export: results, figures, provenance, pre-registration

Everything the paper consumes, in one directory. `prereg.json` is the Gate G1 scaffold: the rules,
their hash, the placeholder flags, and the config hash of the run that exercised them.

In [ ]:
summary = {
    "config_hash": sha(asdict(CFG)), "run_mode": CFG.run_mode, "prereg_hash": PREREG_HASH,
    "gate_G0_pass": bool(G0_PASS),
    "components_estimate": {k: (round(v, 5) if isinstance(v, float) else v) for k, v in est.items() if not k.startswith("_")},
    "components_reference": {k: (round(v, 5) if isinstance(v, float) else v) for k, v in ref.items() if not k.startswith("_")},
    "within_by_rung": {k: round(v, 5) for k, v in wt.items()},
    "ablation_battery_pass_rate": float(ablation_battery["pass"].mean()),
    "budgets": {"detect_0.05_L0": min_budget_for(0.05, within_L0), "detect_0.05_L2": min_budget_for(0.05, within_L2)},
}
with open(f"{RESULTS_DIR}/summary.json", "w") as f: json.dump(summary, f, indent=2, default=str)
recovery.to_csv(f"{RESULTS_DIR}/recovery.csv", index=False)
ablation_battery.to_csv(f"{RESULTS_DIR}/ablation_battery.csv", index=False)
adjudication_final.to_csv(f"{RESULTS_DIR}/adjudication_final.csv", index=False)
if len(h5_df): h5_df.to_csv(f"{RESULTS_DIR}/h5_paired_trace.csv", index=False)
if 'catalog' in dir() and len(catalog): catalog.to_csv(f"{RESULTS_DIR}/detection_catalog.csv", index=False)
if len(flip_emp): flip_emp.to_csv(f"{RESULTS_DIR}/empirical_flip.csv", index=False)
if MODEB.get("est"):
    with open(f"{RESULTS_DIR}/modeb_decomposition.json", "w") as f:
        json.dump({k: (v if not isinstance(v, dict) else {kk: round(vv, 5) for kk, vv in v.items()})
                   for k, v in MODEB["est"].items() if k != "_within_by_rung"} |
                  {"within_by_rung": {k: round(v, 5) for k, v in MODEB["est"]["_within_by_rung"].items()}},
                  f, indent=2, default=str)
prereg_out = {"rules": PREREG, "prereg_hash": PREREG_HASH,
              "placeholders_to_lock_at_G1": ["surveyed_median_delta", "surveyed_delta_quartiles",
                                             "pilot-calibrated (p) thresholds"],
              "config": {k: v for k, v in asdict(CFG).items() if k != "api_key"},
              "config_hash": sha(asdict(CFG)),
              "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}
with open(f"{RESULTS_DIR}/prereg.json", "w") as f: json.dump(prereg_out, f, indent=2, default=str)
print("wrote:", sorted(os.listdir(RESULTS_DIR)))
print(); print("GATE G0:", "PASS" if G0_PASS else "FAIL",
      "| battery pass rate:", f"{ablation_battery['pass'].mean():.0%}",
      "| results in", os.path.abspath(RESULTS_DIR))